# 11 — Trace a run
CrewAI emits execution events. This local listener records event types and
times without capturing prompts or document contents. CrewAI's full tracing
integration can be enabled separately when the organization approves it.



## Problem

A final answer alone cannot explain tool choice or execution order.


## Concept

CrewAI events expose execution milestones; a local listener records a minimal trace.

Events reveal what ran and in what order. A small local trace is enough to diagnose a missing tool call or unexpected retry without logging the entire customer prompt.


## Abstraction

Crew/LLM/tool events → local JSONL trace → diagnosis. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
Crew start ──▶ LLM events ──▶ tool events ──▶ Crew end
      │               │                │              │
      └───────────────┴────────────────┴──────────────┘
                         local JSONL trace
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Register the local CrewAI event listener before kickoff

Register the local CrewAI event listener before kickoff.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from workshop.common import ensure_runtime
from workshop.crew import build_research_crew
from workshop.observability import LocalTrace

listener = LocalTrace()
result = build_research_crew().kickoff(inputs={
    "question": "Should Northstar AI pilot Meridian AI?",
    "customer_context": "No saved customer constraint.",
})



## Read the local trace
Find the Crew start/end, model calls, and tool events. Compare the event
sequence with the report to diagnose a missing tool or unexpected retry.



## Step 2 — Read the JSONL event sequence and locate the first unexpected step

Read the JSONL event sequence and locate the first unexpected step.


In [ ]:
print(result.raw)
print((ensure_runtime() / "trace.jsonl").read_text(encoding="utf-8")[-3000:])


## Execution flow

Crew/LLM/tool events → local JSONL trace → diagnosis. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

A trace that logs full prompts can leak private documents. This local listener records event types and timestamps; record sensitive payloads only under an explicit data policy.


## Production implication

Record correlation IDs, duration, and boundary outcomes while redacting customer data. Decide whether external trace export is permitted before enabling it.


## Your turn

Find the first model event and each tool event in the trace.

**Checkpoint:** You can reconstruct the key run sequence without reading hidden prompts.
